<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Manual_logic_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manual Backtest Logic Checker

This checker always loads the **current `build_excel_grid_table()` and `run_grid_backtest()` functions from `Grid_trading.ipynb` on `main`**. It also reads the current main backtest parameters (`BACKTEST_CAPITAL`, `BACKTEST_GAP`, `BUY_FEE`, `SELL_FEE`, `PRICE_ROUNDING`) directly from the main notebook.

So if the main engine or these parameters are changed later, rerunning this notebook from the top validates the latest version automatically.

`BACKTEST_FLOOR` and `BACKTEST_CEILING` are not fixed constants in the main notebook; they are derived from historical data. For manual testing, boundaries therefore remain manual (or can be derived from the manual OHLC using the same rounding parameter).

## 1. Load the current engine and parameters from the main notebook

In [ ]:
import ast
import bisect
import hashlib
import heapq

import requests
import numpy as np
import pandas as pd

MAIN_NOTEBOOK_URL = (
    'https://raw.githubusercontent.com/'
    'natdanaiii/Trading/main/Grid_trading.ipynb'
)

response = requests.get(MAIN_NOTEBOOK_URL, timeout=30)
response.raise_for_status()
main_notebook = response.json()

def _cell_source(cell):
    source = cell.get('source', '')
    return ''.join(source) if isinstance(source, list) else source

def load_main_function(function_name):
    namespace = {
        'np': np,
        'pd': pd,
        'heapq': heapq,
        'bisect': bisect,
    }

    for cell in main_notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue

        source = _cell_source(cell)
        if f'def {function_name}(' not in source:
            continue

        tree = ast.parse(source)
        for node in tree.body:
            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):
                module = ast.Module(body=[node], type_ignores=[])
                ast.fix_missing_locations(module)
                exec(
                    compile(module, 'Grid_trading.ipynb', 'exec'),
                    namespace,
                )
                return namespace[function_name], ast.unparse(node)

    raise RuntimeError(
        f'Function {function_name!r} was not found in main notebook.'
    )

def _numeric_literal(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if (
        isinstance(node, ast.UnaryOp)
        and isinstance(node.op, ast.USub)
        and isinstance(node.operand, ast.Constant)
        and isinstance(node.operand.value, (int, float))
    ):
        return -float(node.operand.value)
    return None

def extract_main_numeric_assignments():
    values = {}

    for cell in main_notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue

        source = _cell_source(cell)
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:
            if isinstance(node, ast.Assign):
                value = _numeric_literal(node.value)
                if value is None:
                    continue
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        values[target.id] = value
            elif isinstance(node, ast.AnnAssign):
                if not isinstance(node.target, ast.Name):
                    continue
                value = _numeric_literal(node.value)
                if value is not None:
                    values[node.target.id] = value

    return values

build_excel_grid_table, grid_function_source = (
    load_main_function('build_excel_grid_table')
)
run_grid_backtest, engine_function_source = (
    load_main_function('run_grid_backtest')
)

main_assignments = extract_main_numeric_assignments()

MAIN_PARAMETER_MAP = {
    'capital': 'BACKTEST_CAPITAL',
    'gap': 'BACKTEST_GAP',
    'buy_fee': 'BUY_FEE',
    'sell_fee': 'SELL_FEE',
    'price_rounding': 'PRICE_ROUNDING',
}

missing = [
    variable
    for variable in MAIN_PARAMETER_MAP.values()
    if variable not in main_assignments
]
if missing:
    raise RuntimeError(
        'Could not read these main parameters: ' + ', '.join(missing)
    )

main_parameters = {
    name: main_assignments[variable]
    for name, variable in MAIN_PARAMETER_MAP.items()
}

source_hash = hashlib.sha256(
    (
        grid_function_source
        + engine_function_source
        + repr(main_parameters)
    ).encode('utf-8')
).hexdigest()[:16]

print('Loaded CURRENT engine directly from Grid_trading.ipynb')
print(f'Engine + parameter hash: {source_hash}')
print()
print('===== CURRENT MAIN PARAMETERS =====')
for name, value in main_parameters.items():
    print(f'{name:16s}: {value}')

## 2. Manual input

Normally you only edit this cell.

By default, Capital / Gap / Buy Fee / Sell Fee are copied from the **current main notebook**. Put a number in `MANUAL_OVERRIDES` only when you intentionally want to test a different value, e.g. Capital = 1,000.

For boundaries:
- `BOUNDARY_MODE = 'manual'` uses the manual Floor / Ceiling below.
- `BOUNDARY_MODE = 'main_rounding_from_manual_ohlc'` derives Floor / Ceiling from your manual OHLC using the current main `PRICE_ROUNDING`.

Each `MANUAL_OHLC` row is `(Open, High, Low, Close)`.

In [ ]:
# ============================================================
# MANUAL INPUT — EDIT HERE
# ============================================================

# None = use current value from Grid_trading.ipynb
MANUAL_OVERRIDES = {
    'capital': None,       # Example: 1000.0
    'gap': None,
    'buy_fee': None,
    'sell_fee': None,
}

BOUNDARY_MODE = 'manual'

# Used only when BOUNDARY_MODE = 'manual'.
# Keep (ceiling - floor) divisible by the active Gap.
MANUAL_FLOOR = 108000.0
MANUAL_CEILING = 112000.0

# Each row = (Open, High, Low, Close)
# Current example: crosses BUY 111,000, then SELL 112,000.
MANUAL_OHLC = [
    (111500.0, 111600.0, 110500.0, 110800.0),
    (110800.0, 112100.0, 110800.0, 112000.0),
]

MANUAL_START_TIME = '2024-01-01 00:00:00+00:00'

## 3. Resolve active parameters

This table shows exactly which values came from the main notebook and which values you overrode manually.

In [ ]:
active_parameters = {
    key: (
        main_parameters[key]
        if MANUAL_OVERRIDES[key] is None
        else float(MANUAL_OVERRIDES[key])
    )
    for key in ['capital', 'gap', 'buy_fee', 'sell_fee']
}

manual_times = pd.date_range(
    MANUAL_START_TIME,
    periods=len(MANUAL_OHLC),
    freq='min',
    tz='UTC',
)

df_manual_price = pd.DataFrame({
    'open_time': manual_times,
    'open': [x[0] for x in MANUAL_OHLC],
    'high': [x[1] for x in MANUAL_OHLC],
    'low': [x[2] for x in MANUAL_OHLC],
    'close': [x[3] for x in MANUAL_OHLC],
})

if BOUNDARY_MODE == 'manual':
    manual_floor = float(MANUAL_FLOOR)
    manual_ceiling = float(MANUAL_CEILING)

elif BOUNDARY_MODE == 'main_rounding_from_manual_ohlc':
    rounding = float(main_parameters['price_rounding'])
    manual_low = float(df_manual_price['low'].min())
    manual_high = float(df_manual_price['high'].max())
    manual_floor = (
        np.floor(manual_low / rounding) * rounding
    )
    manual_ceiling = (
        np.ceil(manual_high / rounding) * rounding
    )

else:
    raise ValueError(
        "BOUNDARY_MODE must be 'manual' or "
        "'main_rounding_from_manual_ohlc'."
    )

manual_parameters = {
    'capital': active_parameters['capital'],
    'ceiling': manual_ceiling,
    'floor': manual_floor,
    'gap': active_parameters['gap'],
    'buy_fee': active_parameters['buy_fee'],
    'sell_fee': active_parameters['sell_fee'],
}

parameter_rows = []
for key in ['capital', 'gap', 'buy_fee', 'sell_fee']:
    override = MANUAL_OVERRIDES[key]
    parameter_rows.append({
        'parameter': key,
        'main_value': main_parameters[key],
        'manual_override': override,
        'active_value': manual_parameters[key],
        'source': 'MAIN' if override is None else 'OVERRIDE',
    })

parameter_rows.extend([
    {
        'parameter': 'floor',
        'main_value': 'data-derived',
        'manual_override': MANUAL_FLOOR if BOUNDARY_MODE == 'manual' else None,
        'active_value': manual_floor,
        'source': BOUNDARY_MODE,
    },
    {
        'parameter': 'ceiling',
        'main_value': 'data-derived',
        'manual_override': MANUAL_CEILING if BOUNDARY_MODE == 'manual' else None,
        'active_value': manual_ceiling,
        'source': BOUNDARY_MODE,
    },
])

df_parameter_source = pd.DataFrame(parameter_rows)
display(df_parameter_source)

print(
    'Main PRICE_ROUNDING: '
    f'{main_parameters["price_rounding"]:,.6f}'
)
print(f'Boundary mode: {BOUNDARY_MODE}')

## 4. Run the CURRENT real engine with your manual OHLC

In [ ]:
df_manual_grid = build_excel_grid_table(
    **manual_parameters
)

manual_result = run_grid_backtest(
    df_price=df_manual_price,
    grid_table=df_manual_grid,
    initial_capital=manual_parameters['capital'],
)

manual_summary = manual_result['summary']
df_manual_trade_log = manual_result['trade_log']
df_manual_completed = manual_result['completed_trades']
df_manual_equity = manual_result['equity_curve']
df_manual_state = manual_result['grid_state']

print('===== MANUAL BACKTEST LOGIC CHECKER =====')
print(f'Engine + parameter hash : {source_hash}')
print()
print('--- ACTIVE PARAMETERS ---')
print(f'Capital            : {manual_parameters["capital"]:,.6f} USDT')
print(f'Ceiling            : {manual_parameters["ceiling"]:,.6f}')
print(f'Floor              : {manual_parameters["floor"]:,.6f}')
print(f'Gap                : {manual_parameters["gap"]:,.6f}')
print(f'Buy Fee            : {manual_parameters["buy_fee"]:.4%}')
print(f'Sell Fee           : {manual_parameters["sell_fee"]:.4%}')
print(f'Number of Grids    : {len(df_manual_grid)}')
print(f'Capital / Grid     : {df_manual_grid["capital_per_level"].iloc[0]:,.6f} USDT')

print()
print('--- FINAL ENGINE OUTPUT ---')
print(f'Final Cash         : {manual_summary["final_cash"]:,.6f} USDT')
print(f'Final BTC          : {manual_summary["final_btc"]:.12f} BTC')
print(f'Final Equity       : {manual_summary["final_equity"]:,.6f} USDT')
print(f'Realized Profit    : {manual_summary["realized_profit"]:,.6f} USDT')
print(f'Unrealized P&L     : {manual_summary["unrealized_pnl"]:,.6f} USDT')
print(f'Completed Cycles   : {manual_summary["completed_cycles"]}')
print(f'Open Positions     : {manual_summary["open_positions"]}')

print()
print('1) Manual OHLC input')
display(df_manual_price)

print('2) Grid generated by CURRENT main grid function')
display(df_manual_grid[[
    'level',
    'buy_price',
    'sell_price',
    'capital_per_level',
    'gross_base_amount',
    'buy_fee_base',
    'base_amount',
    'gross_sell',
    'sell_fee_quote',
    'net_sell',
    'profit',
]])

print('3) Trade log generated by CURRENT main backtest engine')
if len(df_manual_trade_log):
    display(df_manual_trade_log[[
        'event_id',
        'time',
        'side',
        'grid_level',
        'price',
        'quote_amount',
        'base_amount',
        'fee_base',
        'fee_quote',
        'cash_movement',
        'grid_cashflow',
        'cash_before',
        'cash_after',
        'btc_before',
        'btc_after',
    ]])
else:
    print('No trade was executed.')

print('4) Completed BUY -> SELL cycles')
if len(df_manual_completed):
    display(df_manual_completed[[
        'grid_level',
        'buy_time',
        'sell_time',
        'buy_price',
        'sell_price',
        'cost',
        'base_amount',
        'actual_earn',
        'grid_cashflow',
    ]])
else:
    print('No completed cycle.')

print('5) Portfolio after each manual candle')
display(df_manual_equity[[
    'open_time',
    'close',
    'cash',
    'btc',
    'equity',
    'drawdown',
]])

print('6) Grid positions still open at the end')
manual_open_grids = df_manual_state.loc[
    df_manual_state['holding']
]
if len(manual_open_grids):
    display(manual_open_grids[[
        'level',
        'buy_price',
        'sell_price',
        'capital_per_level',
        'base_amount',
        'buy_time',
    ]])
else:
    print('No open grid position.')